# PDF to Markdown Converter with Image Extraction

This notebook extracts text and images from PDF files and converts them to Markdown format.

**Features:**
- Extract text content from PDFs
- Extract and save embedded images
- Convert to structured Markdown format
- Handle multi-page documents
- Preserve document structure (headings, paragraphs, lists)

**Perfect for running on Google Colab with GPU acceleration for OCR tasks!**

## Step 1: Install Required Libraries

In [ ]:
# Install required packages
!pip install -q pymupdf
!pip install -q PyMuPDF
!pip install -q pdf2image
!pip install -q pytesseract
!pip install -q Pillow
!pip install -q pdfplumber
!pip install -q opencv-python

# For advanced OCR (optional, GPU-accelerated)
!pip install -q easyocr

print("✅ All packages installed successfully!")

## Step 2: Import Libraries

In [ ]:
import fitz  # PyMuPDF
import os
import io
from pathlib import Path
from PIL import Image
import json
from datetime import datetime
import re

print("✅ Libraries imported successfully!")

## Step 3: Mount Google Drive (Optional)

If you want to read PDFs from or save results to Google Drive, run this cell.

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

print("✅ Google Drive mounted at /content/drive")

## Step 4: Upload PDF File

Choose one method:
- **Method A**: Upload from your computer
- **Method B**: Use a file from Google Drive (uncomment the drive path)

In [ ]:
# Method A: Upload from computer
from google.colab import files
uploaded = files.upload()
pdf_path = list(uploaded.keys())[0]

# Method B: Use from Google Drive (uncomment if using)
# pdf_path = '/content/drive/MyDrive/your-pdf-file.pdf'

print(f"✅ PDF file ready: {pdf_path}")

## Step 5: PDF to Markdown Converter Class

In [ ]:
class PDFToMarkdownConverter:
    def __init__(self, pdf_path, output_dir="output"):
        """
        Initialize the PDF to Markdown converter.
        
        Args:
            pdf_path (str): Path to the PDF file
            output_dir (str): Directory to save output files
        """
        self.pdf_path = pdf_path
        self.output_dir = output_dir
        self.images_dir = os.path.join(output_dir, "images")
        
        # Create output directories
        os.makedirs(self.output_dir, exist_ok=True)
        os.makedirs(self.images_dir, exist_ok=True)
        
        # Open PDF
        self.doc = fitz.open(pdf_path)
        self.image_counter = 0
        
    def clean_text(self, text):
        """Clean and normalize extracted text."""
        # Remove excessive whitespace
        text = re.sub(r'\s+', ' ', text)
        # Remove strange characters
        text = text.strip()
        return text
    
    def find_figure_caption(self, page, img_bbox):\n        \"\"\"\n        Find the caption text near an image, looking for FIGURE patterns.\n        \n        Args:\n            page: PyMuPDF page object\n            img_bbox: Image bounding box (x0, y0, x1, y1)\n            \n        Returns:\n            tuple: (figure_number, caption_text) or (None, None)\n        \"\"\"\n        # Get all text blocks on the page\n        blocks = page.get_text(\"dict\")[\"blocks\"]\n        \n        # Define search area (below and around the image)\n        img_x0, img_y0, img_x1, img_y1 = img_bbox\n        search_height = 150  # Search up to 150 points below image\n        \n        caption_candidates = []\n        \n        for block in blocks:\n            if block[\"type\"] == 0:  # Text block\n                block_bbox = block[\"bbox\"]\n                block_x0, block_y0, block_x1, block_y1 = block_bbox\n                \n                # Check if block is near the image (below or above)\n                vertical_distance = abs(block_y0 - img_y1)\n                horizontal_overlap = min(img_x1, block_x1) - max(img_x0, block_x0)\n                \n                if vertical_distance < search_height and horizontal_overlap > 0:\n                    # Extract text from block\n                    text = \"\"\n                    for line in block.get(\"lines\", []):\n                        for span in line.get(\"spans\", []):\n                            text += span.get(\"text\", \"\") + \" \"\n                    \n                    text = text.strip()\n                    \n                    # Look for FIGURE pattern\n                    figure_pattern = r'FIGURE\\s+([\\d\\-]+[A-Z]?)'\n                    match = re.search(figure_pattern, text, re.IGNORECASE)\n                    \n                    if match:\n                        caption_candidates.append({\n                            'text': text,\n                            'figure_num': match.group(1),\n                            'distance': vertical_distance,\n                            'y_pos': block_y0\n                        })\n        \n        # Return the closest caption\n        if caption_candidates:\n            closest = min(caption_candidates, key=lambda x: x['distance'])\n            return closest['figure_num'], closest['text']\n        \n        return None, None\n    \n    def is_figure_image(self, img_bbox, page_width, page_height, min_size=10000):\n        \"\"\"\n        Determine if an image is likely a figure (not decorative).\n        \n        Args:\n            img_bbox: Image bounding box\n            page_width: Page width\n            page_height: Page height\n            min_size: Minimum area in pixels squared\n            \n        Returns:\n            bool: True if image is likely a figure\n        \"\"\"\n        x0, y0, x1, y1 = img_bbox\n        width = x1 - x0\n        height = y1 - y0\n        area = width * height\n        \n        # Filter out very small images (likely decorative)\n        if area < min_size:\n            return False\n        \n        # Filter out very thin images (likely borders/lines)\n        aspect_ratio = max(width, height) / min(width, height)\n        if aspect_ratio > 20:\n            return False\n        \n        return True\n    \n    def extract_images_from_page(self, page, page_num):\n        \"\"\"\n        Extract only figure images from a PDF page with their captions.\n        \n        Args:\n            page: PyMuPDF page object\n            page_num (int): Page number\n            \n        Returns:\n            list: List of dictionaries with image info\n        \"\"\"\n        image_list = page.get_images(full=True)\n        extracted_figures = []\n        \n        page_width = page.rect.width\n        page_height = page.rect.height\n        \n        for img_index, img in enumerate(image_list):\n            xref = img[0]\n            \n            try:\n                # Get image position on page\n                img_rects = page.get_image_rects(xref)\n                if not img_rects:\n                    continue\n                \n                img_bbox = img_rects[0]  # Use first occurrence\n                \n                # Check if this is a figure (not decorative)\n                if not self.is_figure_image(img_bbox, page_width, page_height):\n                    print(f\"  ⏭️  Skipping small/decorative image on page {page_num}\")\n                    continue\n                \n                # Look for figure caption\n                figure_num, caption = self.find_figure_caption(page, img_bbox)\n                \n                if not figure_num:\n                    print(f\"  ⏭️  Skipping image without FIGURE caption on page {page_num}\")\n                    continue\n                \n                # Extract image\n                base_image = self.doc.extract_image(xref)\n                image_bytes = base_image[\"image\"]\n                image_ext = base_image[\"ext\"]\n                \n                # Generate filename based on figure number\n                self.image_counter += 1\n                safe_fig_num = figure_num.replace('-', '_').replace(' ', '_')\n                image_filename = f\"figure_{safe_fig_num}.{image_ext}\"\n                image_path = os.path.join(self.images_dir, image_filename)\n                \n                # Save image\n                with open(image_path, \"wb\") as img_file:\n                    img_file.write(image_bytes)\n                \n                extracted_figures.append({\n                    'filename': image_filename,\n                    'figure_num': figure_num,\n                    'caption': caption,\n                    'page': page_num\n                })\n                \n                print(f\"  ✅ Extracted FIGURE {figure_num}: {image_filename}\")\n                print(f\"     Caption: {caption[:80]}...\")\n                \n            except Exception as e:\n                print(f\"  ⚠️ Error processing image {img_index} on page {page_num}: {e}\")\n        \n        return extracted_figures
    
    def detect_heading_level(self, block):
        """
        Detect if text block is a heading based on font size.
        
        Args:
            block: Text block from PyMuPDF
            
        Returns:
            int: Heading level (1-3) or 0 for normal text
        """
        if 'lines' in block and len(block['lines']) > 0:
            # Get font size from first span
            first_line = block['lines'][0]
            if 'spans' in first_line and len(first_line['spans']) > 0:
                font_size = first_line['spans'][0].get('size', 0)
                
                # Determine heading level based on font size
                if font_size > 18:
                    return 1
                elif font_size > 14:
                    return 2
                elif font_size > 12:
                    return 3
        
        return 0
    
    def extract_text_from_page(self, page):
        """
        Extract text from a page with structure detection.
        
        Args:
            page: PyMuPDF page object
            
        Returns:
            str: Formatted markdown text
        """
        blocks = page.get_text("dict")["blocks"]
        markdown_text = ""
        
        for block in blocks:
            if block["type"] == 0:  # Text block
                text = ""
                for line in block.get("lines", []):
                    for span in line.get("spans", []):
                        text += span.get("text", "")
                    text += " "
                
                text = self.clean_text(text)
                
                if text:
                    # Detect if it's a heading
                    heading_level = self.detect_heading_level(block)
                    
                    if heading_level > 0:
                        markdown_text += f"\n{'#' * heading_level} {text}\n\n"
                    else:
                        markdown_text += f"{text}\n\n"
        
        return markdown_text
    
    def convert_to_markdown(self):
        """
        Convert entire PDF to Markdown with images.
        
        Returns:
            str: Path to the generated markdown file
        """
        print(f"\n🚀 Starting conversion of: {self.pdf_path}")
        print(f"📄 Total pages: {len(self.doc)}\n")
        
        markdown_content = f"# {Path(self.pdf_path).stem}\n\n"
        markdown_content += f"*Extracted on: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}*\n\n"
        markdown_content += "---\n\n"
        
        # Process each page
        for page_num in range(len(self.doc)):
            print(f"📖 Processing page {page_num + 1}/{len(self.doc)}...")
            
            page = self.doc[page_num]
            
            # Add page marker
            markdown_content += f"\n## Page {page_num + 1}\n\n"
            
            # Extract text\n            text = self.extract_text_from_page(page)\n            markdown_content += text\n            \n            # Extract figure images with captions\n            figures = self.extract_images_from_page(page, page_num + 1)\n            \n            # Add figure references to markdown\n            if figures:\n                markdown_content += \"\\n### Figures\\n\\n\"\n                for fig in figures:\n                    markdown_content += f\"**FIGURE {fig['figure_num']}**\\n\\n\"\n                    markdown_content += f\"![Figure {fig['figure_num']}](images/{fig['filename']})\\n\\n\"\n                    markdown_content += f\"*{fig['caption']}*\\n\\n\"\n                    markdown_content += \"---\\n\\n\"\n            \n            markdown_content += \"\\n---\\n\"
        
        # Save markdown file
        output_filename = Path(self.pdf_path).stem + ".md"
        output_path = os.path.join(self.output_dir, output_filename)
        
        with open(output_path, "w", encoding="utf-8") as f:
            f.write(markdown_content)
        
        print(f"\n✅ Conversion complete!")
        print(f"📝 Markdown file: {output_path}")
        print(f"📷 Total images extracted: {self.image_counter}")
        print(f"📁 Images directory: {self.images_dir}")
        
        return output_path
    
    def generate_summary(self):
        """
        Generate a summary JSON file with extraction statistics.
        """
        summary = {
            "pdf_file": self.pdf_path,
            "total_pages": len(self.doc),
            "total_images": self.image_counter,
            "output_directory": self.output_dir,
            "extraction_date": datetime.now().isoformat(),
        }
        
        summary_path = os.path.join(self.output_dir, "extraction_summary.json")
        with open(summary_path, "w", encoding="utf-8") as f:
            json.dump(summary, f, indent=2)
        
        return summary
    
    def close(self):
        """Close the PDF document."""
        self.doc.close()

print("✅ PDFToMarkdownConverter class defined!")

## Step 6: Run the Conversion

In [ ]:
# Configure output directory
output_directory = "pdf_extraction_output"

# Create converter instance
converter = PDFToMarkdownConverter(pdf_path, output_dir=output_directory)

# Convert PDF to Markdown
markdown_file = converter.convert_to_markdown()

# Generate summary
summary = converter.generate_summary()

# Close the PDF
converter.close()

print("\n" + "="*50)
print("📊 EXTRACTION SUMMARY")
print("="*50)
for key, value in summary.items():
    print(f"{key}: {value}")

## Step 7: Preview the Markdown File

In [ ]:
# Read and display first 2000 characters of the markdown file
with open(markdown_file, "r", encoding="utf-8") as f:
    content = f.read()

print("📝 MARKDOWN PREVIEW (first 2000 characters):")
print("="*70)
print(content[:2000])
print("\n...")
print(f"\n(Full content length: {len(content)} characters)")

## Step 8: Display Sample Images

In [ ]:
import matplotlib.pyplot as plt
from PIL import Image
import glob

# Get all extracted images
image_files = glob.glob(os.path.join(output_directory, "images", "*"))

if image_files:
    # Display up to 6 sample images
    num_images = min(6, len(image_files))
    fig, axes = plt.subplots(2, 3, figsize=(15, 10))
    axes = axes.flatten()
    
    for idx, img_path in enumerate(image_files[:num_images]):
        img = Image.open(img_path)
        axes[idx].imshow(img)
        axes[idx].set_title(os.path.basename(img_path), fontsize=10)
        axes[idx].axis('off')
    
    # Hide unused subplots
    for idx in range(num_images, 6):
        axes[idx].axis('off')
    
    plt.tight_layout()
    plt.show()
    
    print(f"\n📷 Displayed {num_images} of {len(image_files)} extracted images")
else:
    print("⚠️ No images were extracted from the PDF.")

## Step 9: Download Results

Download the markdown file and images to your computer.

In [ ]:
import shutil

# Create a zip file with all results
zip_filename = "pdf_extraction_results"
shutil.make_archive(zip_filename, 'zip', output_directory)

print(f"📦 Created zip file: {zip_filename}.zip")

# Download the zip file
from google.colab import files
files.download(f"{zip_filename}.zip")

print("✅ Download started! Check your browser's download folder.")

## Optional: Advanced OCR with EasyOCR (GPU-Accelerated)

If your PDF contains images with text that needs OCR, use this cell.

In [ ]:
# Install EasyOCR if not already installed
try:
    import easyocr
except ImportError:
    !pip install easyocr
    import easyocr

# Initialize EasyOCR reader (uses GPU if available)
reader = easyocr.Reader(['en'], gpu=True)

def ocr_image(image_path):
    """
    Perform OCR on an image using EasyOCR.
    """
    results = reader.readtext(image_path)
    text = ' '.join([result[1] for result in results])
    return text

# Process all extracted images with OCR
print("🔍 Running OCR on extracted images...\n")
ocr_results = {}

for img_file in image_files[:5]:  # Process first 5 images as example
    print(f"Processing: {os.path.basename(img_file)}")
    text = ocr_image(img_file)
    ocr_results[os.path.basename(img_file)] = text
    print(f"  Extracted text: {text[:100]}...\n")

print("✅ OCR processing complete!")

## Cleanup (Optional)

Remove temporary files if needed.

In [ ]:
# Uncomment to clean up temporary files
# import shutil
# shutil.rmtree(output_directory)
# print("🧹 Cleaned up temporary files!")